# AIC2026 — Build Object index theo mapping

Input cần gắn:

- `object_detection_example/object`: JSON Object dạng `<video_id>/<frame>.json`.
- `aic_mapping/artifacts/clip_row_mapping.jsonl`: mapping đang dùng với FAISS.
- `aic2026_keyframe_new/final_dataset/keyframes`: ảnh vật lý để xác minh tên frame.
- `update_config/object_aliases.vi.json`: alias Object tiếng Việt.
- model `update-script-v2`: code pipeline mới nhất.

Notebook ghép Object bằng đúng cặp **`video_id + keyframe_name`**. Object dư không có trong mapping được bỏ qua; nếu thiếu bất kỳ Object nào mà mapping yêu cầu, notebook dừng trước khi build.


In [1]:
from pathlib import Path
import os

INPUT_ROOT = Path("/kaggle/input")
PROJECT_ROOT = Path("/kaggle/working/aic_practice")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"

for directory in (PROJECT_ROOT, SCRIPTS_DIR, ARTIFACTS_DIR, DATA_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

mounted_roots = []
for category in ("datasets", "models", "notebooks"):
    category_root = INPUT_ROOT / category
    if not category_root.exists():
        continue
    for owner_root in category_root.iterdir():
        if not owner_root.is_dir():
            continue
        for mounted_root in owner_root.iterdir():
            if mounted_root.is_dir():
                mounted_roots.append(mounted_root)

print("CÁC INPUT ĐÃ GẮN:")
for root in mounted_roots:
    print("-", root)

CÁC INPUT ĐÃ GẮN:
- /kaggle/input/datasets/blonton/object-detection-example
- /kaggle/input/datasets/blonton/aic-mapping
- /kaggle/input/datasets/blonton/update-config
- /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new
- /kaggle/input/models/blonton/update-script-v2


In [2]:
def normalize_name(value):
    return str(value).casefold().replace("_", "-").replace(" ", "-")


def find_mount(*keywords, required=True):
    normalized = [normalize_name(keyword) for keyword in keywords]
    matches = [
        root
        for root in mounted_roots
        if all(keyword in normalize_name(root) for keyword in normalized)
    ]
    if not matches:
        if required:
            raise FileNotFoundError(f"Không tìm thấy Kaggle Input: {keywords}")
        return None
    if len(matches) > 1:
        print(f"Nhiều Input khớp {keywords}:")
        for match in matches:
            print(" -", match)
    return matches[0]


OBJECT_DATASET_ROOT = find_mount("object", "detection", "example")
CONFIG_DATASET_ROOT = find_mount("update", "config")
MAPPING_DATASET_ROOT = find_mount("aic", "mapping")
KEYFRAME_DATASET_ROOT = find_mount("aic2026", "keyframe", "new")
CODE_DATASET_ROOT = find_mount("update", "script")

RAW_SOURCE_ROOT = OBJECT_DATASET_ROOT / "object"
FINAL_DATASET_ROOT = KEYFRAME_DATASET_ROOT / "final_dataset"
KEYFRAME_SOURCE_ROOT = FINAL_DATASET_ROOT / "keyframes"

assert RAW_SOURCE_ROOT.is_dir(), f"Thiếu {RAW_SOURCE_ROOT}"
assert KEYFRAME_SOURCE_ROOT.is_dir(), f"Thiếu {KEYFRAME_SOURCE_ROOT}"

print("Object raw:", RAW_SOURCE_ROOT)
print("Mapping:", MAPPING_DATASET_ROOT)
print("Keyframe mới:", KEYFRAME_SOURCE_ROOT)
print("Config:", CONFIG_DATASET_ROOT)
print("Code:", CODE_DATASET_ROOT)

Object raw: /kaggle/input/datasets/blonton/object-detection-example/object
Mapping: /kaggle/input/datasets/blonton/aic-mapping
Keyframe mới: /kaggle/input/datasets/ltnngoc/aic2026-keyframe-new/final_dataset/keyframes
Config: /kaggle/input/datasets/blonton/update-config
Code: /kaggle/input/models/blonton/update-script-v2


In [3]:
import shutil


def find_file(root, filename, max_depth=14):
    root = Path(root)
    root_depth = len(root.parts)
    for current, directories, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - root_depth
        if depth >= max_depth:
            directories[:] = []
        if filename in files:
            return current_path / filename
    return None


BUILD_SCRIPT_SOURCE = find_file(CODE_DATASET_ROOT, "build_object_index.py")
MAPPING_SOURCE = find_file(MAPPING_DATASET_ROOT, "clip_row_mapping.jsonl")
ALIAS_SOURCE = find_file(CONFIG_DATASET_ROOT, "object_aliases.vi.json")

assert BUILD_SCRIPT_SOURCE, "Model code thiếu build_object_index.py"
assert MAPPING_SOURCE, "aic_mapping thiếu clip_row_mapping.jsonl"
assert ALIAS_SOURCE, "update_config thiếu object_aliases.vi.json"

CODE_SOURCE_DIR = BUILD_SCRIPT_SOURCE.parent
for required in (
    "build_object_index.py",
    "object_io.py",
    "object_label_catalog.py",
    "object_retriever.py",
    "search_types.py",
    "stdio_setup.py",
):
    assert (CODE_SOURCE_DIR / required).is_file(), f"Code thiếu {required}"

for source in CODE_SOURCE_DIR.glob("*.py"):
    shutil.copy2(source, SCRIPTS_DIR / source.name)

MAPPING_PATH = ARTIFACTS_DIR / "clip_row_mapping.jsonl"
ALIASES_PATH = CONFIG_DIR / "object_aliases.vi.json"
shutil.copy2(MAPPING_SOURCE, MAPPING_PATH)
shutil.copy2(ALIAS_SOURCE, ALIASES_PATH)

print("✅ Code:", CODE_SOURCE_DIR)
print("✅ Mapping:", MAPPING_PATH)
print("✅ Aliases:", ALIASES_PATH)

✅ Code: /kaggle/input/models/blonton/update-script-v2/pytorch/default/1
✅ Mapping: /kaggle/working/aic_practice/artifacts/clip_row_mapping.jsonl
✅ Aliases: /kaggle/working/aic_practice/config/object_aliases.vi.json


## Đối chiếu mapping, keyframe vật lý và Object theo tên frame


In [4]:
from collections import defaultdict
from pathlib import Path
import json
import os

# ============================================================
# 1. ĐỌC MAPPING
# ============================================================

mapping_by_video = defaultdict(list)

with MAPPING_PATH.open("r", encoding="utf-8-sig") as file:
    for line in file:
        if not line.strip():
            continue

        row = json.loads(line)
        mapping_by_video[str(row["video_id"])].append(row)

for video_id in mapping_by_video:
    mapping_by_video[video_id].sort(
        key=lambda row: int(row["vector_index"])
    )


# ============================================================
# 2. ĐỌC OBJECT JSON
# ============================================================

raw_by_video = defaultdict(list)

for video_dir in RAW_SOURCE_ROOT.iterdir():
    try:
        if not video_dir.is_dir():
            continue

        paths = [
            path
            for path in video_dir.glob("*.json")
            if path.stem.isdigit()
        ]

        paths.sort(
            key=lambda path: int(path.stem)
        )

        if paths:
            raw_by_video[video_dir.name] = paths

    except OSError as e:
        print(
            f"[WARNING] Không đọc được object folder "
            f"{video_dir.name}: {e}"
        )
        continue


# ============================================================
# 3. ĐỌC DANH SÁCH ẢNH VẬT LÝ
# ============================================================

physical_by_video = {}

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

scan_errors = []

for video_dir in KEYFRAME_SOURCE_ROOT.iterdir():

    # Kiểm tra folder
    try:
        if not video_dir.is_dir():
            continue
    except OSError as e:
        print(
            f"[WARNING] Không kiểm tra được folder "
            f"{video_dir}: {e}"
        )
        continue

    names = []

    try:
        # Không dùng entry.is_file()
        # vì Kaggle mount có thể lỗi khi gọi stat metadata
        with os.scandir(video_dir) as entries:

            for entry in entries:
                try:
                    name = entry.name
                    p = Path(name)

                    # Chỉ lấy đúng extension ảnh
                    if p.suffix.casefold() not in image_extensions:
                        continue

                    # Chỉ lấy file có tên dạng số:
                    # 000001.jpg, 015225.jpg, ...
                    if not p.stem.isdigit():
                        continue

                    names.append(name)

                except OSError as e:
                    scan_errors.append(
                        (
                            video_dir.name,
                            getattr(entry, "name", "UNKNOWN"),
                            str(e),
                        )
                    )

                    print(
                        f"[WARNING] Lỗi entry trong "
                        f"{video_dir.name}: {e}"
                    )

                    continue

    except OSError as e:
        scan_errors.append(
            (
                video_dir.name,
                None,
                str(e),
            )
        )

        print(
            f"[WARNING] Không scan được folder "
            f"{video_dir.name}: {e}"
        )

        continue

    # Sort theo số frame
    names.sort(
        key=lambda name: int(Path(name).stem)
    )

    if names:
        physical_by_video[video_dir.name] = names


# ============================================================
# 4. THỐNG KÊ
# ============================================================

mapping_rows = sum(
    len(rows)
    for rows in mapping_by_video.values()
)

raw_documents = sum(
    len(paths)
    for paths in raw_by_video.values()
)

physical_images = sum(
    len(names)
    for names in physical_by_video.values()
)


print(
    "Mapping:",
    len(mapping_by_video),
    "video /",
    f"{mapping_rows:,}",
    "keyframe",
)

print(
    "Object:",
    len(raw_by_video),
    "video /",
    f"{raw_documents:,}",
    "JSON",
)

print(
    "Ảnh vật lý:",
    len(physical_by_video),
    "video /",
    f"{physical_images:,}",
    "ảnh",
)

print(
    "Filesystem scan errors:",
    len(scan_errors),
)


# Chỉ hiện tối đa 20 lỗi đầu tiên
if scan_errors:
    print("\nMột số lỗi scan:")

    for error in scan_errors[:20]:
        print(" -", error)

Mapping: 873 video / 196,590 keyframe
Object: 873 video / 201,998 JSON
Ảnh vật lý: 873 video / 196,590 ảnh
Filesystem scan errors: 0


In [5]:
# ============================================================
# VALIDATION THEO video_id + keyframe_name
#
# Ví dụ:
# mapping: L21_V001 / 000030.jpg
# object:  L21_V001 / 000030.json
#
# Object dư được ghi nhận và bỏ qua. Object thiếu sẽ chặn build.
# ============================================================

all_video_ids = sorted(
    set(mapping_by_video)
    | set(raw_by_video)
    | set(physical_by_video)
)

videos_without_mapping = []
missing_objects = []
extra_objects = []
physical_name_mismatches = []
object_lookup_by_video = {}

for video_id in all_video_ids:
    mapping_rows_for_video = mapping_by_video.get(video_id, [])
    object_paths = raw_by_video.get(video_id, [])
    physical_names = physical_by_video.get(video_id, [])

    object_by_stem = {path.stem: path for path in object_paths}
    object_lookup_by_video[video_id] = object_by_stem

    if object_paths and not mapping_rows_for_video:
        videos_without_mapping.append(video_id)
        continue

    mapping_names = [
        str(row["keyframe_name"])
        for row in mapping_rows_for_video
    ]
    expected_stems = {
        Path(name).stem
        for name in mapping_names
    }
    actual_stems = set(object_by_stem)

    missing_stems = sorted(
        expected_stems - actual_stems,
        key=int,
    )
    extra_stems = sorted(
        actual_stems - expected_stems,
        key=int,
    )

    if missing_stems:
        missing_objects.append({
            "video_id": video_id,
            "count": len(missing_stems),
            "names": [f"{stem}.json" for stem in missing_stems[:30]],
        })

    if extra_stems:
        extra_objects.append({
            "video_id": video_id,
            "count": len(extra_stems),
            "names": [f"{stem}.json" for stem in extra_stems[:30]],
        })

    if physical_names and mapping_names != physical_names:
        physical_name_mismatches.append(video_id)

missing_object_count = sum(item["count"] for item in missing_objects)
extra_object_count = sum(item["count"] for item in extra_objects)
matched_object_count = mapping_rows - missing_object_count

print("=" * 70)
print("VALIDATION THEO video_id + keyframe_name")
print("=" * 70)
print(f"Object khớp mapping: {matched_object_count:,}/{mapping_rows:,}")
print(f"Object thiếu: {missing_object_count:,} thuộc {len(missing_objects)} video")
print(f"Object dư sẽ bỏ qua: {extra_object_count:,} thuộc {len(extra_objects)} video")
print(f"Video Object không có mapping: {len(videos_without_mapping)}")
print(
    "Mapping name lệch ảnh vật lý (WARNING):",
    len(physical_name_mismatches),
    physical_name_mismatches[:20],
)

if missing_objects:
    print("\nMột số Object còn thiếu:")
    for item in missing_objects[:30]:
        print(
            f" - {item['video_id']}: thiếu {item['count']}, "
            f"ví dụ={item['names']}"
        )

if extra_objects:
    print("\nMột số Object dư (sẽ bỏ qua):")
    for item in extra_objects[:30]:
        print(
            f" - {item['video_id']}: dư {item['count']}, "
            f"ví dụ={item['names']}"
        )

assert not videos_without_mapping, (
    "Không được build: có thư mục Object nhưng không có video tương ứng "
    "trong clip_row_mapping"
)
assert not missing_objects, (
    f"Không được build: thiếu {missing_object_count:,} Object JSON "
    "mà clip_row_mapping yêu cầu"
)

print("\n✅ Tất cả Object cần thiết đều khớp chính xác theo tên frame.")
print("✅ Object dư sẽ không được đưa vào index.")


VALIDATION THEO video_id + keyframe_name
Object khớp mapping: 196,590/196,590
Object thiếu: 0 thuộc 0 video
Object dư sẽ bỏ qua: 5,408 thuộc 48 video
Video Object không có mapping: 0
Mapping name lệch ảnh vật lý (WARNING): 0 []

Một số Object dư (sẽ bỏ qua):
 - L26_V185: dư 123, ví dụ=['000325.json', '000350.json', '000375.json', '000400.json', '000650.json', '000675.json', '000700.json', '000725.json', '000750.json', '000775.json', '000825.json', '000850.json', '000875.json', '000900.json', '000925.json', '000975.json', '001000.json', '001025.json', '001050.json', '001100.json', '001125.json', '001150.json', '001175.json', '001200.json', '001225.json', '001250.json', '001300.json', '001325.json', '001350.json', '001375.json']
 - L26_V186: dư 105, ví dụ=['000350.json', '000375.json', '000625.json', '000650.json', '000675.json', '000700.json', '000725.json', '000750.json', '000800.json', '000825.json', '000850.json', '000875.json', '000900.json', '000925.json', '000975.json', '001000.js

In [6]:
import sys

sys.path.insert(0, str(SCRIPTS_DIR))
from object_io import parse_detections

sample_paths = [path for video_id in sorted(raw_by_video) for path in raw_by_video[video_id]][:20]
for path in sample_paths:
    raw = json.loads(path.read_text(encoding="utf-8-sig"))
    detections = parse_detections(raw, source=str(path))
    print(path.relative_to(RAW_SOURCE_ROOT), "→", len(detections), "detection")

print("✅ Schema Object đọc được")

L21_V001/000000.json → 38 detection
L21_V001/000030.json → 8 detection
L21_V001/000060.json → 21 detection
L21_V001/000090.json → 24 detection
L21_V001/000120.json → 25 detection
L21_V001/000360.json → 16 detection
L21_V001/000390.json → 20 detection
L21_V001/000420.json → 51 detection
L21_V001/000450.json → 46 detection
L21_V001/000480.json → 20 detection
L21_V001/000510.json → 10 detection
L21_V001/000570.json → 24 detection
L21_V001/000630.json → 42 detection
L21_V001/000660.json → 36 detection
L21_V001/000690.json → 43 detection
L21_V001/000720.json → 41 detection
L21_V001/000780.json → 4 detection
L21_V001/000840.json → 14 detection
L21_V001/000870.json → 12 detection
L21_V001/000900.json → 15 detection
✅ Schema Object đọc được


## Build Object SQLite

Builder được monkey-patch trong kernel để lấy trực tiếp Object có cùng `video_id` và stem của `keyframe_name`. Raw Dataset và code model không bị sửa; file Object dư được bỏ qua.


In [7]:
import hashlib
from functools import lru_cache

import build_object_index as object_builder
from object_io import ObjectDocument


@lru_cache(maxsize=1024)
def load_object_payload(path_string):
    path = Path(path_string)
    raw = json.loads(path.read_text(encoding="utf-8-sig"))
    canonical = json.dumps(
        raw,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return raw, hashlib.sha256(canonical).hexdigest()


def iter_aligned_object_documents(inputs, on_error=None):
    produced = 0
    video_ids = sorted(
        mapping_by_video,
        key=lambda video_id: int(mapping_by_video[video_id][0]["vector_index"]),
    )

    for video_id in video_ids:
        object_by_stem = object_lookup_by_video[video_id]

        for mapping_row in mapping_by_video[video_id]:
            keyframe_stem = Path(str(mapping_row["keyframe_name"])).stem
            source_path = object_by_stem[keyframe_stem]
            raw, content_hash = load_object_payload(str(source_path))

            yield ObjectDocument(
                source=f"keyframe_name_exact:{source_path}",
                logical_id=f"{video_id}/{keyframe_stem}",
                video_id=video_id,
                keyframe_stem=keyframe_stem,
                raw=raw,
                content_sha256=content_hash,
            )

            produced += 1
            if produced % 50000 == 0:
                print(f"Đã ánh xạ {produced:,} Object document...", flush=True)


object_builder.iter_object_documents = iter_aligned_object_documents

OUTPUT_INDEX = ARTIFACTS_DIR / "object_index.sqlite3"
object_builder.build(
    inputs=[RAW_SOURCE_ROOT],
    output_path=OUTPUT_INDEX,
    mapping_path=MAPPING_PATH,
    min_score=0.4,
    manifest_path=None,
    allow_unmapped=False,
    require_complete=True,
)
print("✅ Build xong:", OUTPUT_INDEX)


Đã index 10,000 Object keyframe...
Đã index 20,000 Object keyframe...
Đã index 30,000 Object keyframe...
Đã index 40,000 Object keyframe...
Đã index 50,000 Object keyframe...
Đã ánh xạ 50,000 Object document...
Đã index 60,000 Object keyframe...
Đã index 70,000 Object keyframe...
Đã index 80,000 Object keyframe...
Đã index 90,000 Object keyframe...
Đã index 100,000 Object keyframe...
Đã ánh xạ 100,000 Object document...
Đã index 110,000 Object keyframe...
Đã index 120,000 Object keyframe...
Đã index 130,000 Object keyframe...
Đã index 140,000 Object keyframe...
Đã index 150,000 Object keyframe...
Đã ánh xạ 150,000 Object document...
Đã index 160,000 Object keyframe...
Đã index 170,000 Object keyframe...
Đã index 180,000 Object keyframe...
Đã index 190,000 Object keyframe...
Đã tạo Object SQLite index
  Mapping: 196,590 keyframe
  Object đã map: 196,590
  Object còn thiếu: 0
  Object không map được: 0
  Nhãn: 503
  Detection được index: 905,974
  Hoàn chỉnh: có
  Output: /kaggle/working

In [8]:
import sqlite3
from datetime import datetime

connection = sqlite3.connect(OUTPUT_INDEX)
alignment_metadata = {
    "alignment_mode": "video_id_and_keyframe_name_exact",
    "alignment_verified": "video_id_keyframe_name_and_physical_names",
    "exact_detection": "1",
    "exact_video_count": str(len(mapping_by_video)),
    "approximate_video_count": "0",
    "source_object_documents": str(raw_documents),
    "mapped_source_object_documents": str(mapping_rows),
    "ignored_extra_object_documents": str(extra_object_count),
    "keyframe_dataset": "aic2026_keyframe_new/final_dataset",
    "source_dataset": "object_detection_example/object",
    "aligned_at": datetime.now().isoformat(),
}
connection.executemany(
    "INSERT OR REPLACE INTO metadata(key, value) VALUES (?, ?)",
    alignment_metadata.items(),
)
connection.commit()
connection.close()
print("✅ Đã ghi metadata căn chỉnh chính xác theo tên frame")


✅ Đã ghi metadata căn chỉnh chính xác theo tên frame


In [9]:
connection = sqlite3.connect(OUTPUT_INDEX)
metadata = dict(connection.execute("SELECT key, value FROM metadata").fetchall())
document_count = connection.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
posting_count = connection.execute("SELECT COUNT(*) FROM postings").fetchone()[0]
class_count = connection.execute("SELECT COUNT(*) FROM classes").fetchone()[0]
distinct_sources = connection.execute("SELECT COUNT(DISTINCT source) FROM documents").fetchone()[0]
connection.close()

actual_mapping_hash = hashlib.sha256(MAPPING_PATH.read_bytes()).hexdigest()
expected_alignment_mode = "video_id_and_keyframe_name_exact"

print("Documents:", f"{document_count:,}")
print("Postings:", f"{posting_count:,}")
print("Classes:", class_count)
print("Nguồn Object đã dùng:", f"{distinct_sources:,}/{raw_documents:,}")
print("Object dư đã bỏ qua:", f"{extra_object_count:,}")
for key in (
    "schema", "mapping_rows", "mapped_object_documents",
    "missing_object_documents", "unmapped_object_documents",
    "complete", "retrieval_ready", "alignment_mode", "exact_detection",
    "exact_video_count", "approximate_video_count",
    "ignored_extra_object_documents",
):
    print(f"{key}: {metadata.get(key)}")

assert metadata.get("mapping_sha256") == actual_mapping_hash
assert document_count == mapping_rows
assert distinct_sources == mapping_rows
assert metadata.get("complete") == "1"
assert metadata.get("retrieval_ready") == "1"
assert metadata.get("alignment_mode") == expected_alignment_mode
assert metadata.get("exact_detection") == "1"
assert metadata.get("ignored_extra_object_documents") == str(extra_object_count)
print("✅ Object index khớp chính xác mapping mới")


Documents: 196,590
Postings: 582,360
Classes: 503
Nguồn Object đã dùng: 196,590/201,998
Object dư đã bỏ qua: 5,408
schema: aic_object_sqlite_v1
mapping_rows: 196590
mapped_object_documents: 196590
missing_object_documents: 0
unmapped_object_documents: 0
complete: 1
retrieval_ready: 1
alignment_mode: video_id_and_keyframe_name_exact
exact_detection: 1
exact_video_count: 873
approximate_video_count: 0
ignored_extra_object_documents: 5408
✅ Object index khớp chính xác mapping mới


In [10]:
import csv

REPORT_PATH = Path("/kaggle/working/object_build_report.csv")
with REPORT_PATH.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=(
            "video_id",
            "mapping_count",
            "object_count",
            "physical_count",
            "matched_count",
            "missing_count",
            "ignored_extra_count",
            "alignment_mode",
        ),
    )
    writer.writeheader()

    for video_id in sorted(mapping_by_video):
        expected_stems = {
            Path(str(row["keyframe_name"])).stem
            for row in mapping_by_video[video_id]
        }
        actual_stems = set(object_lookup_by_video[video_id])

        writer.writerow({
            "video_id": video_id,
            "mapping_count": len(expected_stems),
            "object_count": len(actual_stems),
            "physical_count": len(physical_by_video.get(video_id, [])),
            "matched_count": len(expected_stems & actual_stems),
            "missing_count": len(expected_stems - actual_stems),
            "ignored_extra_count": len(actual_stems - expected_stems),
            "alignment_mode": "video_id_and_keyframe_name_exact",
        })

print("✅ Báo cáo:", REPORT_PATH)


✅ Báo cáo: /kaggle/working/object_build_report.csv


## Đóng gói Dataset Object cho notebook tổng

In [11]:
import zipfile

EXPORT_ROOT = Path("/kaggle/working/aic2026_object_ready")
EXPORT_ARTIFACTS = EXPORT_ROOT / "artifacts"
EXPORT_CONFIG = EXPORT_ROOT / "config"
EXPORT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
EXPORT_CONFIG.mkdir(parents=True, exist_ok=True)

shutil.copy2(OUTPUT_INDEX, EXPORT_ARTIFACTS / "object_index.sqlite3")
shutil.copy2(MAPPING_PATH, EXPORT_ARTIFACTS / "clip_row_mapping.jsonl")
shutil.copy2(ALIASES_PATH, EXPORT_CONFIG / "object_aliases.vi.json")
shutil.copy2(REPORT_PATH, EXPORT_ROOT / REPORT_PATH.name)

ZIP_PATH = Path("/kaggle/working/aic2026_object_ready.zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in EXPORT_ROOT.rglob("*"):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(EXPORT_ROOT))

print("✅ Dataset folder:", EXPORT_ROOT)
print("✅ ZIP:", ZIP_PATH)
print("Dung lượng:", round(ZIP_PATH.stat().st_size / 1024**2, 2), "MB")

✅ Dataset folder: /kaggle/working/aic2026_object_ready
✅ ZIP: /kaggle/working/aic2026_object_ready.zip
Dung lượng: 21.96 MB


In [12]:
from IPython.display import FileLink, display

display(
    FileLink(
        str(ZIP_PATH),
        result_html_prefix="Nhấn để tải Object index: ",
    )
)

/kaggle/working/aic2026_object_ready.zip